# Player Status and News Notebook

This notebook is an exploratory workflow for building reliable player-availability signals from MLB StatsAPI.

## Why this version exists

If `/transactions` only returns a narrow set of types in your date window, it may not be enough for fantasy roster management by itself. This notebook now probes multiple related endpoints and compares what each endpoint can (and cannot) tell us.


In [ ]:
from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path
import importlib.util
import json
import urllib.parse
import urllib.request
import urllib.error

import pandas as pd


## Configuration


In [ ]:
SEASON = date.today().year
LOOKBACK_DAYS = 14
SPORT_ID = 1

START_DATE = date(SEASON - 1, 1, 1)
END_DATE = date.today()
SNAPSHOT_DATE = END_DATE
DATE_TAG = SNAPSHOT_DATE.isoformat().replace("-", "")

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = "https://statsapi.mlb.com/api/v1"


## Request helpers (debug-friendly)

These helpers preserve URL, HTTP status, and an error preview so call-shape problems are easy to diagnose.


In [ ]:
def build_url(path: str, params: dict | None = None) -> str:
    params = params or {}
    query = urllib.parse.urlencode(params)
    url = f"{BASE_URL}{path}"
    return f"{url}?{query}" if query else url


def fetch_json_debug(path: str, params: dict | None = None, timeout: int = 60) -> dict:
    url = build_url(path, params)
    result = {
        "ok": False,
        "url": url,
        "status": None,
        "error": None,
        "payload": None,
        "response_text_preview": None,
    }

    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            result["status"] = response.status
            raw_text = response.read().decode("utf-8")
            result["response_text_preview"] = raw_text[:500]
            result["payload"] = json.loads(raw_text)
            result["ok"] = True
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        result["status"] = exc.code
        result["error"] = f"HTTPError: {exc}"
        result["response_text_preview"] = body[:500]
    except urllib.error.URLError as exc:
        result["error"] = f"URLError: {exc}"
    except json.JSONDecodeError as exc:
        result["error"] = f"JSONDecodeError: {exc}"

    return result


def payload_records(payload: dict | None, keys: list[str]) -> list[dict]:
    if not payload:
        return []
    for key in keys:
        if key in payload and isinstance(payload[key], list):
            return payload[key]
    return []


## Step 1: connectivity sanity check


In [ ]:
sports_check = fetch_json_debug("/sports", {"sportId": SPORT_ID})

print("OK:", sports_check["ok"])
print("Status:", sports_check["status"])
print("URL:", sports_check["url"])
print("Error:", sports_check["error"])
print("Preview:", sports_check["response_text_preview"])


## Step 2: endpoint options to evaluate

Use this as a shortlist of endpoints that can contribute to injury / availability status.


In [ ]:
endpoint_catalog = pd.DataFrame([
    {
        "purpose": "Historical movement log",
        "endpoint": "/transactions",
        "example_params": "sportId, startDate, endDate, transactionTypes",
        "strength": "Best historical source when type codes are recognized",
        "limitation": "May miss statuses if wrong type codes or if event is not represented as a transaction"
    },
    {
        "purpose": "Discover valid transaction codes",
        "endpoint": "/transactionTypes",
        "example_params": "sportId",
        "strength": "Lets you verify exact server-supported type codes/descriptions",
        "limitation": "Reference metadata only, not player events"
    },
    {
        "purpose": "Live injured list snapshot",
        "endpoint": "/teams/{teamId}/roster",
        "example_params": "rosterType=injured&date=YYYY-MM-DD",
        "strength": "Direct current IL membership for a team/date",
        "limitation": "Snapshot-oriented; historical backfill requires repeated date pulls"
    },
    {
        "purpose": "Live active/40-man context",
        "endpoint": "/teams/{teamId}/roster",
        "example_params": "rosterType=active or 40Man",
        "strength": "Compares injured vs active vs 40-man status",
        "limitation": "Not a transaction history by itself"
    },
    {
        "purpose": "Player-level roster context",
        "endpoint": "/people/{personId}",
        "example_params": "hydrate=currentTeam,rosterEntries(team)",
        "strength": "Adds player context and roster-entry metadata",
        "limitation": "Not a complete replacement for transactions"
    },
])

endpoint_catalog


## Step 3: discover transaction type metadata


In [ ]:
transaction_types_check = fetch_json_debug("/transactionTypes", {"sportId": SPORT_ID})

transaction_type_records = payload_records(transaction_types_check.get("payload"), ["transactionTypes"])
transaction_type_df = pd.json_normalize(transaction_type_records) if transaction_type_records else pd.DataFrame()

print("Transaction type endpoint OK:", transaction_types_check["ok"])
print("Rows:", len(transaction_type_df))
transaction_type_df.head(30)


## Step 4: transaction probes

Use only an unfiltered transaction pull (`type_code=ALL` behavior) because explicit typed probes excluded many real-world transaction categories in testing.


In [ ]:
common_params = {
    "sportId": SPORT_ID,
    "startDate": START_DATE.isoformat(),
    "endDate": END_DATE.isoformat(),
}

unfiltered_transactions = fetch_json_debug("/transactions", common_params)

unfiltered_records = payload_records(unfiltered_transactions.get("payload"), ["transactions"])

call_log = pd.DataFrame([
    {
        "probe": "unfiltered",
        "type_code": "ALL",
        "ok": unfiltered_transactions["ok"],
        "status": unfiltered_transactions["status"],
        "row_count": len(unfiltered_records),
        "url": unfiltered_transactions["url"],
        "error": unfiltered_transactions["error"],
    }
])

call_log


In [ ]:
frames = []

if unfiltered_records:
    base_frame = pd.json_normalize(unfiltered_records)
    base_frame["requested_type_code"] = "ALL"
    frames.append(base_frame)

transactions = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
transactions = transactions.drop_duplicates() if not transactions.empty else transactions

print(f"Loaded {len(transactions):,} unique rows from unfiltered transaction probe.")
transactions.head(10)


## Step 5: roster endpoint probes (injured / active / 40-man) for all active teams

This now iterates every currently active MLB team and pulls roster snapshots for `injured`, `active`, and `40Man` roster types on `SNAPSHOT_DATE`.


In [ ]:
teams_check = fetch_json_debug("/teams", {"sportId": SPORT_ID, "season": SEASON})
team_records = payload_records(teams_check.get("payload"), ["teams"])
teams_df = pd.json_normalize(team_records) if team_records else pd.DataFrame()

if not teams_df.empty and "active" in teams_df.columns:
    active_teams = teams_df.loc[teams_df["active"] == True, ["id", "name"]].copy()
else:
    active_teams = teams_df[["id", "name"]].copy() if not teams_df.empty else pd.DataFrame(columns=["id", "name"])

active_teams = active_teams.sort_values("name").reset_index(drop=True)
print(f"Active teams selected: {len(active_teams)}")
active_teams.head(40)


In [ ]:
roster_probe_rows = []
roster_frames = []
roster_types = ["injured", "active", "40Man"]

for _, team in active_teams.iterrows():
    team_id = int(team["id"])
    team_name = team["name"]
    for roster_type in roster_types:
        params = {"rosterType": roster_type, "date": SNAPSHOT_DATE.isoformat()}
        result = fetch_json_debug(f"/teams/{team_id}/roster", params)
        records = payload_records(result.get("payload"), ["roster"])

        roster_probe_rows.append({
            "snapshot_date": SNAPSHOT_DATE.isoformat(),
            "team_id": team_id,
            "team_name": team_name,
            "roster_type": roster_type,
            "ok": result["ok"],
            "status": result["status"],
            "row_count": len(records),
            "url": result["url"],
            "error": result["error"],
        })

        if records:
            frame = pd.json_normalize(records)
            frame["snapshot_date"] = SNAPSHOT_DATE.isoformat()
            frame["team_id"] = team_id
            frame["team_name"] = team_name
            frame["roster_type"] = roster_type
            roster_frames.append(frame)

roster_probe_log = pd.DataFrame(roster_probe_rows)
roster_snapshot = pd.concat(roster_frames, ignore_index=True) if roster_frames else pd.DataFrame()

roster_probe_log


In [ ]:
if not roster_snapshot.empty:
    roster_view_cols = [
        c for c in [
            "team_name", "roster_type", "person.id", "person.fullName", "status.code", "status.description", "jerseyNumber", "position.abbreviation"
        ]
        if c in roster_snapshot.columns
    ]
    roster_snapshot_view = roster_snapshot[roster_view_cols].sort_values(["team_name", "roster_type", "person.fullName"])
else:
    roster_snapshot_view = pd.DataFrame()

roster_snapshot_view.head(50)


## Step 6: normalize transactions for downstream use


In [ ]:
if not transactions.empty:
    rename_map = {
        "person.id": "player_id",
        "person.fullName": "player_name",
        "toTeam.id": "to_team_id",
        "toTeam.name": "to_team_name",
        "fromTeam.id": "from_team_id",
        "fromTeam.name": "from_team_name",
        "typeCode": "type_code",
        "typeDesc": "type_desc",
        "description": "description",
        "date": "transaction_date",
        "effectiveDate": "effective_date",
    }
    transactions = transactions.rename(columns={k: v for k, v in rename_map.items() if k in transactions.columns})

    for dt_col in ["transaction_date", "effective_date"]:
        if dt_col in transactions.columns:
            transactions[dt_col] = pd.to_datetime(transactions[dt_col], errors="coerce")

transactions.head(10)


## Step 7: recent fantasy-relevant dashboard


In [ ]:
recent_cutoff = pd.Timestamp(date.today() - timedelta(days=LOOKBACK_DAYS))

dashboard_cols = [
    "transaction_date",
    "player_name",
    "type_desc",
    "description",
    "to_team_name",
    "from_team_name",
]
available_dashboard_cols = [c for c in dashboard_cols if c in transactions.columns]

if not transactions.empty and "transaction_date" in transactions.columns:
    recent_status = (
        transactions.loc[transactions["transaction_date"] >= recent_cutoff, available_dashboard_cols]
        .sort_values("transaction_date", ascending=False)
        .reset_index(drop=True)
    )
else:
    recent_status = pd.DataFrame(columns=available_dashboard_cols)

recent_status.head(50)


## Step 8: optional reference to `MLB-StatsAPI` Python wrapper

If installed, this confirms whether the wrapper is available in your environment for cross-checking endpoint names and helper methods.


In [ ]:
statsapi_available = importlib.util.find_spec("statsapi") is not None
print("statsapi package available:", statsapi_available)

if statsapi_available:
    import statsapi
    print("statsapi module path:", statsapi.__file__)


## Save outputs


In [ ]:
transactions_out = DATA_DIR / f"player_transactions_{SEASON}.parquet"
recent_out = DATA_DIR / f"player_status_recent_{SEASON}.parquet"
call_log_out = DATA_DIR / f"player_status_call_log_{SEASON}.parquet"
roster_probe_out = DATA_DIR / f"player_status_roster_probe_{DATE_TAG}.parquet"
roster_snapshot_out = DATA_DIR / f"player_status_roster_snapshot_{DATE_TAG}.parquet"
transaction_type_out = DATA_DIR / f"player_status_transaction_types_{SEASON}.parquet"
active_teams_out = DATA_DIR / f"player_status_active_teams_{DATE_TAG}.parquet"

if not transactions.empty:
    transactions.to_parquet(transactions_out, index=False)

if not recent_status.empty:
    recent_status.to_parquet(recent_out, index=False)

if not call_log.empty:
    call_log.to_parquet(call_log_out, index=False)

if not roster_probe_log.empty:
    roster_probe_log.to_parquet(roster_probe_out, index=False)

if not roster_snapshot.empty:
    roster_snapshot.to_parquet(roster_snapshot_out, index=False)

if not transaction_type_df.empty:
    transaction_type_df.to_parquet(transaction_type_out, index=False)

if not active_teams.empty:
    active_teams.to_parquet(active_teams_out, index=False)

print("Wrote:")
for p in [
    transactions_out,
    recent_out,
    call_log_out,
    roster_probe_out,
    roster_snapshot_out,
    transaction_type_out,
    active_teams_out,
]:
    print(f" - {p}")


## Notes

- The transaction workflow now relies on a single unfiltered `/transactions` pull (`ALL`) to avoid dropping categories due to incomplete type lists.
- Use `transaction_type_df` as reference metadata for interpreting returned `typeCode` / `typeDesc` values.
- `active_teams` captures every currently active team and is used for full-league roster snapshots.
- `roster_snapshot_view` (`rosterType=injured`) is the most direct current injury status signal; reconstructing history still requires repeated dated snapshots.
- Use `statsapi` wrapper as a convenience layer only; raw endpoint probes remain the most reliable debugging source.
